# Mycobacterium tuberculosis Gene Expression Profiles from In Vitro Cultures and Aerosol-Infected Rabbit Lung Tissue Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the Mycobacterium tuberculosis gene expression dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.s19g-aqd5/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata info
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"License: {meta.license}")
print(f"Published: {meta.datePublished}\nIdentifier: {meta.identifier}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema defines record sets, their fields, and columns by unique `@id`. To explore, list all record sets, their field `@id`s, and available columns.

In [ ]:
# Show overview of record sets and their fields
record_sets = dataset.metadata.recordSet  # List of RecordSet objects

if not record_sets:
    print("No record sets found in schema. This dataset may store fields elsewhere or require inspecting the source.")

# If there are record sets, enumerate their ids and field structures.
for idx, rs in enumerate(record_sets):
    print(f"RecordSet {idx} (@id: {rs['@id']}): {rs.get('name', 'No name')}")
    fields = rs.get('field', [])
    if fields:
        for f in fields:
            field_id = f['@id'] if isinstance(f, dict) and '@id' in f else str(f)
            print(f"  Field @id: {field_id}")
    else:
        print("  No fields listed.")
    columns = rs.get('column', [])
    if columns:
        for col in columns:
            col_id = col['@id'] if isinstance(col, dict) and '@id' in col else str(col)
            print(f"  Column @id: {col_id}")
    else:
        print("  No columns listed.")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview. If no record sets, try loading main records.

In [ ]:
# Attempt to extract data from all record sets (if present)
# If none, attempt to load default records

dataframes = {}
record_set_ids = []
record_sets = dataset.metadata.recordSet

if not record_sets or len(record_sets) == 0:
    # Try loading top-level records
    print("No explicit record sets found. Attempting to load main records...")
    records = list(dataset.records())
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records. Columns:\n{df.columns.tolist()}")
    dataframes['default'] = df
    display(df.head())
else:
    for rs in record_sets:
        rs_id = rs['@id']
        record_set_ids.append(rs_id)
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"RecordSet: {rs_id}\nColumns: {df.columns.tolist()}\n")
            display(df.head())
        except Exception as e:
            print(f"Failed to load records for {rs_id}: {e}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filter records based on specific criteria
- Normalize numeric fields
- Categorize or group records

If numeric fields are available, remove outliers, transform distributions, or group by a categorical column.

In [ ]:
# Choose which DataFrame to analyze
if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]
    print(f"Analyzing DataFrame '{df_key}' with {df.shape[0]} rows.")

    # Attempt to identify numeric fields (by dtype or by column name)
    numeric_columns = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_columns:
        # Try guessing from column names (e.g. 'expression', 'value', 'intensity')
        for col in df.columns:
            if any(s in col.lower() for s in ['expression','value','intensity','count','fdr','fold','score']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if pd.api.types.is_numeric_dtype(df[col]):
                        numeric_columns.append(col)
                except:
                    pass
    print(f"Numeric columns found: {numeric_columns}")

    # Pick the first numeric column for EDA
    if numeric_columns:
        numeric_field = numeric_columns[0]
        # Filter rows where numeric_field > threshold
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} ({filtered_df.shape[0]} rows):")
        display(filtered_df.head())

        # Normalize that numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a categorical field
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No DataFrame available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For gene expression datasets, common visualizations include histogram of expression values, boxplot by sample/source, or scatter of fold-change vs FDR.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]

    # Try to visualize the numeric field
    numeric_columns = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_columns:
        numeric_field = numeric_columns[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break

        if group_field:
            plt.figure(figsize=(10,5))
            sns.boxplot(data=df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=30)
            plt.show()

        # Try a scatter if there is another numeric column
        if len(numeric_columns) > 1:
            plt.figure(figsize=(7,5))
            sns.scatterplot(data=df, x=numeric_columns[0], y=numeric_columns[1])
            plt.title(f"Scatter: {numeric_columns[0]} vs {numeric_columns[1]}")
            plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides transcriptomic profiles of *Mycobacterium tuberculosis* from both in vitro cultures and rabbit lung tissue.
- Data fields accessible via Croissant `@id` allow standardized exploration.
- Numeric fields (expression values, fold-change, etc.) can be filtered, normalized, and grouped for insights.
- Visualizations reveal distributions and differences across sample groups.

Further analysis could investigate specific gene expression patterns, compare in vitro/in vivo conditions, or integrate phenotype metadata from columns referenced by `@id`.